In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
os.chdir('/content/drive/MyDrive/NLP/Translation')
!ls

'arabic dataset'			    mbert_final
'CELAL ŞAMİL KARTOĞLU MAKALE TASLAK.docx'   merged_datasets
'chinese dataset'			    sentiment_gpt3_train.jsonl
 Datasets.ipynb				   'Translation v1.ipynb'
'Datasets v2.ipynb'			   'Translation v2.ipynb'
 final_models				   'Translation v3.ipynb'
'french dataset'			   'Translation v4.ipynb'
 get-pip.py				   'Translation v5.ipynb'
 get-pip.py.1				    xlmr_best_model
'italian dataset'			    xlmr_confusion_matrix.png
 mbert_confusion_matrix.png


In [ ]:
base_dir = "/content/drive/MyDrive/NLP/Translation"
final_models_dir = os.path.join(base_dir, "final_models")
merged_dir = os.path.join(base_dir, "merged_datasets")

print("Base dir exists:", os.path.exists(base_dir))
print("final_models exists:", os.path.exists(final_models_dir))
print("merged_datasets exists:", os.path.exists(merged_dir))

print("\nfinal_models içeriği:")
!ls "/content/drive/MyDrive/NLP/Translation/final_models"

print("\nmerged_datasets içeriği:")
!ls "/content/drive/MyDrive/NLP/Translation/merged_datasets" | head -30

Base dir exists: True
final_models exists: True
merged_datasets exists: True

final_models içeriği:
bertweet_final	      deberta_final	   roberta_libre_final
bertweet_libre_final  deberta_libre_final  roberta_mbart_final
bertweet_mbart_final  roberta_final

merged_datasets içeriği:
arabic_merged_3class_clean_final.csv
arabic_merged_3class.csv
arabic_merged_clean.csv
arabic_merged.csv
chinese_merged_3class_clean_final.csv
chinese_merged_3class.csv
chinese_merged.csv
french_merged_3class_clean_final.csv
french_merged_3class.csv
italian_merged_3class_clean_final.csv
italian_merged_3class.csv
merged_all_languages_clean.csv
merged_balanced.csv
merged_balanced_translated_google_clean.csv
merged_balanced_translated_libre_fixed.csv
merged_balanced_translated_mbart_clean.csv


In [ ]:
import pandas as pd

data_path = "/content/drive/MyDrive/NLP/Translation/merged_datasets/merged_balanced_translated_google_clean.csv"

df = pd.read_csv(data_path)

print("Shape:", df.shape)
df.head()

Shape: (11997, 4)


,text,sentiment,language,translated_text
0,بطلنا نحكم على الناس من شكلهم و لبسهم و تمن خر...,NEG,ar,"We stopped judging people by their looks, thei..."
1,vérifier la quantité à la livraison jai acheté...,NEG,fr,check the quantity on delivery I bought these ...
2,برومو حلقة بائع الليمون ثاني حلقات الأنيميشن م...,NEG,ar,"Promo for the Lemon Seller episode, the second..."
3,il governo fino ad ora non avrà proprio spacca...,POS,it,up to now the government hasn't exactly split ...
4,أثار تخريب كنيسة مارجرجس وحرق بيوت الأقباط بقر...,NEG,ar,The vandalism of St. George's Church and the b...


In [ ]:
texts = df["translated_text"].astype(str).tolist()
labels = df["sentiment"].tolist()

print("Toplam örnek:", len(texts))
print("Örnek text:", texts[0])
print("Örnek label:", labels[0])

Toplam örnek: 11997
Örnek text: We stopped judging people by their looks, their clothes, their outings, and whether they know English or not.
Örnek label: NEG


In [ ]:
label2id = {
    "NEG": 0,
    "NEUTRAL": 1,
    "POS": 2
}

id2label = {v: k for k, v in label2id.items()}

y_true = [label2id[l] for l in labels]

print("İlk 5 label (string):", labels[:5])
print("İlk 5 label (numeric):", y_true[:5])

İlk 5 label (string): ['NEG', 'NEG', 'NEG', 'POS', 'NEG']
İlk 5 label (numeric): [0, 0, 0, 2, 0]


In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

base_model_path = "/content/drive/MyDrive/NLP/Translation/final_models"

model_paths = {
    "deberta": f"{base_model_path}/deberta_final",
    "roberta": f"{base_model_path}/roberta_final",
    "bertweet": f"{base_model_path}/bertweet_final"
}

models = {}
tokenizers = {}

for name, path in model_paths.items():
    print(f"Loading {name}...")

    tokenizers[name] = AutoTokenizer.from_pretrained(path)
    models[name] = AutoModelForSequenceClassification.from_pretrained(path).to(device)
    models[name].eval()

print("Tüm modeller yüklendi ✅")

Device: cuda
Loading deberta...


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

Loading roberta...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading bertweet...


emoji is not installed, thus not converting emoticons or emojis into text. Install emoji: pip3 install emoji==0.6.0


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Tüm modeller yüklendi ✅


In [ ]:
import torch
import numpy as np

def get_probs(text, model, tokenizer):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    ).to(device)

    with torch.no_grad():
        outputs = model(**inputs)

    probs = torch.softmax(outputs.logits, dim=1)
    return probs.squeeze().cpu().numpy()

In [ ]:
sample_text = texts[0]

p_deberta = get_probs(sample_text, models["deberta"], tokenizers["deberta"])
p_roberta = get_probs(sample_text, models["roberta"], tokenizers["roberta"])
p_bertweet = get_probs(sample_text, models["bertweet"], tokenizers["bertweet"])

print("DeBERTa:", p_deberta)
print("RoBERTa:", p_roberta)
print("BERTweet:", p_bertweet)

DeBERTa: [0.9783921  0.01583682 0.00577107]
RoBERTa: [0.25384164 0.7128273  0.03333106]
BERTweet: [0.37085205 0.5886737  0.04047426]


In [ ]:
import numpy as np
from collections import Counter

# ✔ Majority Voting
def majority_voting(probs_list):
    preds = [np.argmax(p) for p in probs_list]
    return Counter(preds).most_common(1)[0][0]


# ✔ Soft Voting (Probability Averaging)
def soft_voting(probs_list):
    avg_probs = np.mean(probs_list, axis=0)
    return np.argmax(avg_probs)


# ✔ Weighted Voting
weights = [0.34, 0.33, 0.33]  # şimdilik eşit gibi

def weighted_voting(probs_list, weights):
    weighted = np.zeros_like(probs_list[0])

    for p, w in zip(probs_list, weights):
        weighted += p * w

    return np.argmax(weighted)

In [ ]:
probs_list = [p_deberta, p_roberta, p_bertweet]

print("Majority:", majority_voting(probs_list))
print("Soft:", soft_voting(probs_list))
print("Weighted:", weighted_voting(probs_list, weights))

Majority: 1
Soft: 0
Weighted: 0


In [ ]:
from tqdm import tqdm

y_pred_majority = []
y_pred_soft = []
y_pred_weighted = []

sample_size = len(texts)

for text in tqdm(texts[:sample_size]):

    probs_list = [
        get_probs(text, models["deberta"], tokenizers["deberta"]),
        get_probs(text, models["roberta"], tokenizers["roberta"]),
        get_probs(text, models["bertweet"], tokenizers["bertweet"])
    ]

    y_pred_majority.append(majority_voting(probs_list))
    y_pred_soft.append(soft_voting(probs_list))
    y_pred_weighted.append(weighted_voting(probs_list, weights))

100%|██████████| 11997/11997 [08:27<00:00, 23.66it/s]


In [ ]:
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, confusion_matrix
import numpy as np

def compute_specificity(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred)

    specificities = []

    for i in range(len(cm)):
        TP = cm[i, i]
        FN = np.sum(cm[i, :]) - TP
        FP = np.sum(cm[:, i]) - TP
        TN = np.sum(cm) - (TP + FN + FP)

        specificity = TN / (TN + FP) if (TN + FP) != 0 else 0
        specificities.append(specificity)

    return np.mean(specificities)


def evaluate(y_true, y_pred, name):
    print(f"\n--- {name} ---")

    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, average="weighted")
    prec = precision_score(y_true, y_pred, average="weighted")
    rec = recall_score(y_true, y_pred, average="weighted")
    spec = compute_specificity(y_true, y_pred)

    print("Accuracy    :", acc)
    print("Precision   :", prec)
    print("Recall      :", rec)
    print("F1-Score    :", f1)
    print("Specificity :", spec)

In [ ]:
y_true_subset = y_true

evaluate(y_true_subset, y_pred_majority, "Majority Voting")
evaluate(y_true_subset, y_pred_soft, "Soft Voting")
evaluate(y_true_subset, y_pred_weighted, "Weighted Voting")


--- Majority Voting ---
Accuracy    : 0.9023922647328498
Precision   : 0.9017953045504747
Recall      : 0.9023922647328498
F1-Score    : 0.9018539702620446
Specificity : 0.9507141009771155

--- Soft Voting ---
Accuracy    : 0.9138117862799033
Precision   : 0.9134281625892896
Recall      : 0.9138117862799033
F1-Score    : 0.9133982049416574
Specificity : 0.9563100283282351

--- Weighted Voting ---
Accuracy    : 0.9141452029674085
Precision   : 0.913768788836889
Recall      : 0.9141452029674085
F1-Score    : 0.9137293706619879
Specificity : 0.9564599226436196
